**Contents**

1. [Setup & Import](#1--Setup--import)
2. [Load Dataset](#2--Datasetload)
3. [PreProceesing & Create Pipeline](#3--Encoading--Standardzation)
4. [Select model](#5--Create-pipeline)
6. [Train and test data with model](#6--Training--testing)
7. [Create pickel file](#7--PickelFile--)


<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Setup & Imports
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        Initialize libraries and environment
    </div>
</div>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import HTML, display 
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score,r2_score,precision_score,recall_score,f1_score,roc_auc_score
import pickle

PALETTE   = "YlGn"
ACCENT    = "#10b981"
DARK      = "#0f2d24"

sns.set_theme(style="whitegrid", palette="YlGn")
plt.rcParams.update({
    "figure.figsize":    (13, 5),
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   DARK,
    "axes.labelsize":    11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.facecolor":  "white",
    "axes.facecolor":    "#fafafa",
    "grid.color":        "#e5e7eb",
})
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

#Data load

<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Load Data
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        load a Dataset and just take look
    </div>
</div>

In [2]:
train_df=pd.read_csv('train.csv')
test_df=pd.read_csv('test.csv')


In [3]:
def summrize_data(df:pd.DataFrame,label:str)->None:
    display(HTML(f"""
    <div style="border-left:5px solid #10b981; padding:10px 18px; background:#f0fdf4;
                border-radius:0 8px 8px 0; margin:12px 0; font-family:'Inter',sans-serif;">
        <b style="color:#065f46; font-size:15px;">{label}</b>
        <span style="margin-left:12px; color:#6b7280; font-size:13px;">
            {df.shape[0]:,} rows &times; {df.shape[1]} columns
        </span>
    </div>
    """))
    display(df.head())
    print()
    
summrize_data(train_df,"Training data")
summrize_data(test_df,  "Test data")


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.9200,32.5800,1.0100,3.0500,15.0100,50.6100,725.9900,5.9000,16.7900,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.8200,No,112.1600,East,Low
1,1,Clay,7.0800,56.6100,0.4400,2.0000,22.9200,67.8600,985.6600,6.9800,3.3900,Wheat,Vegetative,Kharif,Rainfed,River,5.2700,Yes,47.1600,South,Low
2,2,Clay,5.6900,27.7100,0.8100,2.8300,26.9700,92.2200,"2,201.7000",6.0500,3.8500,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.2400,Yes,110.3800,North,Low
3,3,Sandy,5.6500,13.3200,1.3300,0.8700,13.3200,61.5700,"1,357.3300",9.1200,2.3100,Wheat,Flowering,Kharif,Canal,River,8.3200,Yes,53.8500,South,Medium
4,4,Clay,7.9600,59.1400,0.3800,0.9600,20.2200,91.1100,"1,538.2000",6.9500,13.9400,Wheat,Sowing,Rabi,Canal,River,7.3700,No,93.1900,South,Low


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.3600,26.1900,0.5900,2.8100,17.8300,30.2400,"1,533.3800",5.4000,3.0000,Maize,Sowing,Rabi,Canal,River,13.5900,Yes,47.4800,West
1,630001,Clay,5.8700,9.8800,1.1800,3.2600,21.1800,78.0700,576.0500,7.2200,15.8800,Cotton,Sowing,Rabi,Drip,Reservoir,6.1200,Yes,56.4300,South
2,630002,Sandy,6.2200,26.5500,0.9600,0.8500,26.8700,60.3500,545.3000,9.4300,2.6300,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.1100,Yes,20.0000,East
3,630003,Clay,7.6800,53.5800,0.8300,0.5500,41.7400,36.0500,"1,211.0300",6.6900,1.8600,Maize,Harvest,Rabi,Canal,Groundwater,2.2700,No,102.9900,North
4,630004,Loamy,5.2300,59.0200,0.5400,2.1100,41.0800,52.4700,"1,321.9100",4.1100,5.7100,Cotton,Sowing,Kharif,Canal,Groundwater,12.3900,Yes,13.3300,Central


In [4]:
def modify_column(col):
    return col.map({
        "Medium": 2,
        "High": 1,
        "Low": 0
    })

train_df['Irrigation_Need'] = modify_column(train_df['Irrigation_Need'])


<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Data split
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        Spliting data for training 
    </div>
</div>

In [5]:
X=train_df.drop(['Irrigation_Need','id'],axis=1)
y=train_df['Irrigation_Need']


In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Preprocessing & Pipeline Creation
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        Do preprocessing and create pipeline to data train workflow
    </div>
</div>

In [7]:
cat_features=X.select_dtypes(include="object").columns
num_features=X.select_dtypes(exclude='object').columns

num_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

cat_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy="most_frequent")),
    ('Encoding',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor=ColumnTransformer(
    transformers=[
        ('num',num_pipeline,num_features),
        ('cat',cat_pipeline,cat_features)
    ]
)




<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Model Selection
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        Check a which model has best test accuracy compare with training accuracy
    </div>
</div>

In [8]:
models={
    # 'Decision Tree':DecisionTreeClassifier(),
    'RF':RandomForestClassifier(n_estimators=10, max_depth=6),
    # 'XGBOOST':XGBClassifier(),
    # 'ADABOOST':AdaBoostClassifier(),
    # # 'Cat':CatBoostClassifier(verbose=0)
}

In [9]:
model_list = []
accuracy_list=[]

best_accuracy = 0
best_pipe = None

for name, model in models.items():
    
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipe.fit(X_train, y_train)
    
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, y_test_pred)
    
    model_list.append(name)
    accuracy_list.append(acc)
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_pipe = pipe
    print(name)
    
    print("Training Data:")
    print("- Accuracy:", accuracy_score(y_train, y_train_pred))
    print("- F1:", f1_score(y_train, y_train_pred, average='weighted'))
    print("- Precision:", precision_score(y_train, y_train_pred, average='weighted'))
    print("- Recall:", recall_score(y_train, y_train_pred, average='weighted'))
    
    print("Testing Data:")
    print("- Accuracy:", accuracy_score(y_test, y_test_pred))
    print("- F1:", f1_score(y_test, y_test_pred, average='weighted'))
    print("- Precision:", precision_score(y_test, y_test_pred, average='weighted'))
    print("- Recall:", recall_score(y_test, y_test_pred, average='weighted'))
    
    print("="*40)

RF
Training Data:
- Accuracy: 0.9270079365079366
- F1: 0.9190524279865218
- Precision: 0.928156584154654
- Recall: 0.9270079365079366
Testing Data:
- Accuracy: 0.9270793650793651
- F1: 0.919100727074306
- Precision: 0.9281343094417912
- Recall: 0.9270793650793651


In [10]:
results_df = pd.DataFrame({
    'Model': model_list,
    'Accuracy': accuracy_list
})

print(results_df.sort_values(by='Accuracy',ascending=False))

  Model  Accuracy
0    RF    0.9271


In [11]:
best_model_name = results_df.sort_values(by='Accuracy', ascending=False).iloc[0]['Model']
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {best_accuracy:.4f}")

Best Model: RF
Best Accuracy: 0.9271


<div style="
    padding: 16px 22px;
    border-radius: 12px;
    background: #f0fdf4;
    border: 1px solid #bbf7d0;
    margin: 26px 0 12px 0;
    font-family: 'Inter', sans-serif;
">
    <div style="font-size:19px; font-weight:600; color:#065f46;">
        Model file
    </div>
    <div style="font-size:13px; color:#6b7280; margin-top:4px;">
        create model file and import it 
    </div>
</div>

In [12]:
# import pickle
# import joblib

# joblib.dump(best_pipe, "best_model_pipeline.pkl", compress=5)
# print("✅ Compressed model saved")

In [13]:
import joblib

# your trained model variable
joblib.dump(model, "best_model_pipeline.pkl", compress=0)
print("saved successfully")

saved successfully


In [14]:
import os
print(os.path.getsize("best_model_pipeline.pkl") / 1024**2)

0.1116647720336914
